In [1]:
from jqdatasdk import *

auth('18613838075','Rest1129')

提示：当前环境 pandas 版本高于 0.25，get_price 与 get_fundamentals_continuously 接口 panel 参数将固定为 False
注意：0.25 以上版本 pandas 不支持 panel，如使用该数据结构和相关函数请注意修改
auth success 


In [1]:
import sqlite3
import datetime

sqlite_file = './sqlite_data/dp.sqlite'
#sqlite_file = './sqlite_data/dp.sqlite'

# Connecting to the database file
conn = sqlite3.connect(sqlite_file)
c = conn.cursor()

In [3]:
# create tables exe only if needed

create_sql = """
create table if not exists XSHG_000300 (
    date datetime primary key desc,
    open real,
    high real,
    low real,
    close real,
    volume bigint
    ) without rowid
    """

c.execute(create_sql)
conn.commit()

In [4]:
st_date = datetime.date(2000,1,1)

etf_data_df = get_price('000300.XSHG', start_date=st_date, end_date=datetime.date.today(), fq=None)

In [6]:
l = list()

etf_data_df.dropna(inplace=True)

for idx,row in etf_data_df.iterrows():
    l.append(
        ( row.name.date(),
         float(row['open']),
         float(row['high']),
         float(row['low']),
         float(row['close']),
         int(row['volume']),
        )
    )

sql = '''insert into XSHG_000300(date,open,high,low,close,volume) values (?,?,?,?,?,?)'''

c.executemany(sql,(l))
conn.commit()

In [2]:
# convert to csv
query_sql = 'select * from XSHG_000300'
c.execute(query_sql)

dbdata = c.fetchall()
dbdata.reverse()

In [3]:
dbdata

[('2005-04-08', 984.66, 1003.7, 979.53, 1003.45, 1476252600),
 ('2005-04-11', 1003.88, 1008.74, 992.77, 995.42, 1593606600),
 ('2005-04-12', 993.71, 993.71, 978.2, 978.7, 1022619300),
 ('2005-04-13', 987.95, 1006.5, 987.95, 1000.9, 1607168700),
 ('2005-04-14', 1004.64, 1006.42, 985.58, 986.98, 1294571000),
 ('2005-04-15', 982.61, 982.61, 971.93, 974.08, 1040895500),
 ('2005-04-18', 970.91, 970.91, 958.65, 963.77, 859840000),
 ('2005-04-19', 962.92, 968.87, 957.91, 965.89, 921262300),
 ('2005-04-20', 964.15, 964.15, 946.2, 950.87, 885070500),
 ('2005-04-21', 948.86, 955.55, 938.6, 943.98, 994614500),
 ('2005-04-22', 942.91, 947.91, 934.96, 939.1, 1069188400),
 ('2005-04-25', 935.99, 935.99, 920.16, 930.07, 1147047800),
 ('2005-04-26', 928.43, 939.7, 924.65, 937.08, 1169047100),
 ('2005-04-27', 938.57, 938.91, 925.9, 926.6, 1078061100),
 ('2005-04-28', 923.53, 945.5, 914.83, 942.07, 1434348600),
 ('2005-04-29', 940.81, 942.45, 929.81, 932.39, 1123541900),
 ('2005-05-09', 934.65, 937.39, 

In [5]:
import csv

with open('./csv_files/000300_XSHG.csv','w') as f:
    csv_writer = csv.writer(f)
    
    # write header
    
    csv_writer.writerow(('date','open','high','low','close','volume'))
    
    csv_writer.writerows(dbdata)

In [3]:
'''
159915.XSHE 创业
159920.XSHE 恒生 
510300.XSHG 沪深300
515750.XSHG 科技50
518880.XSHG 黄金
'''
etf_list = ['159915.XSHE', '159920.XSHE', '510300.XSHG', '515750.XSHG', '518880.XSHG']

for etf in etf_list:
    print('Begin handle ETF: {}'.format(etf))
    collection = db['etf_'+etf]

    # query last record of etf
    qres = list(collection.find().sort('date',pymongo.DESCENDING).limit(1))

    if qres:
        # already got some data
        last_record = qres[0]
        last_date = last_record['date'].date()
        if last_date == datetime.date.today():
            print('Already updated last date:{}'.format(last_date))
            print('Early Stop handle: {}'.format(etf))
            continue
        st_date = last_record['date'].date() + datetime.timedelta(days=1)
        print('Data found. start_date: {}'.format(st_date))
    else:
        # empty create unique index
        collection.create_index([('date', pymongo.ASCENDING)], unique=True)
        # empty query 10 years 
        st_date = datetime.date(2010,1,1)
        print('Empty. start_date: {}'.format(st_date))
        
    etf_data_df = get_price(etf, start_date=st_date, end_date=datetime.date.today(), fq=None)
    if etf_data_df.empty:
        print('Query data empty Early Stop:{}'.format(etf))
        continue
    etf_data_df.dropna(inplace=True)
    
    print('Query data sucess')
    doclist = []
    for idx,row in etf_data_df.iterrows():
        doclist.append(
            dict(
                date=datetime.datetime.combine(row.name.date(),datetime.time()),
                open=float(row['open']),
                high=float(row['high']),
                low=float(row['low']),
                close=float(row['close']),
                volume=int(row['volume']),
            )
        )
    
    print('Doc list insert many...')
    collection.insert_many(doclist)
    print('End handle ETF: {}'.format(etf))

Begin handle ETF: 159915.XSHE
Data found. start_date: 2020-06-30
Query data sucess
Doc list insert many...
End handle ETF: 159915.XSHE
Begin handle ETF: 159920.XSHE
Data found. start_date: 2020-06-30
Query data sucess
Doc list insert many...
End handle ETF: 159920.XSHE
Begin handle ETF: 510300.XSHG
Data found. start_date: 2020-06-30
Query data sucess
Doc list insert many...
End handle ETF: 510300.XSHG
Begin handle ETF: 515750.XSHG
Data found. start_date: 2020-06-30
Query data sucess
Doc list insert many...
End handle ETF: 515750.XSHG
Begin handle ETF: 518880.XSHG
Data found. start_date: 2020-06-30
Query data sucess
Doc list insert many...
End handle ETF: 518880.XSHG
